# Predicting Sales Prices

 ## Dataset Attributes

 The dataset contains information about houses in Ames. The data was collected to describe property sales which occurred in Ames.

In [ ]:
# Import libraries

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

%matplotlib inline
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (8,5)


In [ ]:
# Read in the data set

df = pd.read_csv('ames.csv')
print("Dataset loaded. Shape:", df.shape)
df.head()

# Quick overview: columns, types, missing values

In [ ]:
# Quick info and check for required columns
required_cols = ['Gr_Liv_Area', 'Garage_Area', 'Sale_Price']
print("Required columns present:", all(col in df.columns for col in required_cols))
display(df[required_cols].info())
display(df[required_cols].describe())
print("\nMissing values in relevant columns:")
print(df[required_cols].isnull().sum())

### Data cleaning plan
- For modeling with the two selected variables I will:
  - Drop rows missing any of `Gr_Liv_Area`, `Garage_Area`, or `Sale_Price`.
  - examine whether zeros in `Garage_Area` meaningfully represent 'no garage'.
- I'll record how many rows are dropped and continue with cleaned data.

In [ ]:
# Inspect rows where Garage_Area is zero
num_zero_garage = (df['Garage_Area'] == 0).sum()
print(f"Rows with Garage_Area == 0: {num_zero_garage}")

# Drop rows with missing values in the relevant columns
before = df.shape[0]
df_clean = df.dropna(subset=required_cols).copy()
after = df_clean.shape[0]
print(f"Dropped {before - after} rows with missing values. Remaining rows: {after}")

## Exploratory Data Analysis (EDA)
I will:
- Visualise distributions of `Sale_Price`, `Gr_Liv_Area`, and `Garage_Area`.
- Explore pairwise relationships and correlations.

In [ ]:
# Explore the data with visualisations such as histograms and correlation matrices
# Distribution plots
fig, axes = plt.subplots(1, 3, figsize=(18,4))
sns.histplot(df_clean['Sale_Price'], kde=True, ax=axes[0]).set_title('Sale_Price distribution')
sns.histplot(df_clean['Gr_Liv_Area'], kde=True, ax=axes[1]).set_title('Gr_Liv_Area distribution')
sns.histplot(df_clean['Garage_Area'], kde=True, ax=axes[2]).set_title('Garage_Area distribution')
plt.show()

# Log-transform Sale_Price sometimes helps visual interpretation
fig, ax = plt.subplots(1,2, figsize=(12,4))
sns.histplot(np.log1p(df_clean['Sale_Price']), kde=True, ax=ax[0]).set_title('Log(Sale_Price + 1)')
sns.boxplot(x=df_clean['Sale_Price'], ax=ax[1]).set_title('Boxplot Sale_Price')
plt.show()

In [ ]:
# Correlation and scatter matrix for the 3 variables
subset = df_clean[required_cols].copy()
print("Correlation matrix:")
display(subset.corr())

sns.pairplot(subset, corner=True)
plt.suptitle("Pairwise scatterplots (Gr_Liv_Area, Garage_Area, Sale_Price)", y=1.02)
plt.show()

### Observations from EDA
**Sale_Price Distribution:**
Highly right-skewed, with most homes priced between $100,000 and $250,000.
A long tail extends toward $700,000+, indicating the presence of high-value outliers.
This skewness violates the normality assumption of linear regression.

**Gr_Liv_Area Distribution:**
Mildly right-skewed.
Most homes fall between 1,000 and 2,000 sq ft, with some extreme large properties above 5,000 sq ft.

**Garage_Area Distribution:**
Multi-modal distribution (peaks around 300, 450, 600 sq ft).
A cluster at 0 sq ft likely represents homes with no garage.
Slight right-skewness due to a few very large garages.

In [ ]:
# Split the independent variables from the dependent variable
X = df_clean[['Gr_Liv_Area', 'Garage_Area']].copy()
y = df_clean['Sale_Price'].copy()

print("X shape:", X.shape, "y shape:", y.shape)
display(X.head())

In [ ]:
# Creating a training and test set with a 75:25 split ratio
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)
print("Training set:", X_train.shape, "Test set:", X_test.shape)

## Build and train the multiple linear regression model
Model: `Sale_Price = intercept + coef1 * Gr_Liv_Area + coef2 * Garage_Area`

In [ ]:
# Build a multiple linear regression model using 'Gr_Liv_Area' and 'Garage_Area'
# Train model
lr = LinearRegression()
lr.fit(X_train, y_train)


In [ ]:
# Print the trained model's intercept and coefficients
intercept = lr.intercept_
coefs = pd.Series(lr.coef_, index=X.columns)

print(f"Intercept: {intercept:.2f}")
print("Coefficients:")
display(coefs)

### Interpretation of coefficients (raw, unstandardised)
- The coefficient for `Gr_Liv_Area` represents the expected change in Sale_Price associated with a one square-foot increase in living area, holding `Garage_Area` constant.
- The coefficient for `Garage_Area` represents the expected change in SalePrice associated with a one square-foot increase in garage area, holding `Gr_Liv_Area` constant.
- I will present numerical values below and interpret them in the context of median sale_price.

In [ ]:
# Generate predictions for the test set
y_pred = lr.predict(X_test)

# Create a small results dataframe
results_df = X_test.copy()
results_df['Actual'] = y_test
results_df['Predicted'] = y_pred
results_df['Residual'] = results_df['Actual'] - results_df['Predicted']
results_df[['Gr_Liv_Area','Garage_Area','Actual','Predicted','Residual']].head()


In [ ]:
# Evaluate the model's performance by computing the mean squared error (MSE) or root mean squared error (RMSE) on the test set using sklearn.metrics.
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print(f"Test MSE: {mse:,.2f}")
print(f"Test RMSE: {rmse:,.2f}")
print(f"Test R^2: {r2:.4f}")


In [ ]:
# Generating an error plot to visualise the differences between the predicted and actual values in the test set.

# Predicted vs Actual scatter
plt.figure(figsize=(8,6))
plt.scatter(y_test, y_pred, alpha=0.6)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', linewidth=2)
plt.xlabel('Actual Sale_Price')
plt.ylabel('Predicted Sale_Price')
plt.title('Predicted vs Actual (Test set)')
plt.show()

# Residual plot (Residuals vs Predicted)
plt.figure(figsize=(8,6))
plt.scatter(y_pred, y_test - y_pred, alpha=0.6)
plt.axhline(0, color='red', linestyle='--')
plt.xlabel('Predicted Sale_Price')
plt.ylabel('Residual (Actual - Predicted)')
plt.title('Residuals vs Predicted (Test set)')
plt.show()

# Histogram of residuals
plt.figure(figsize=(8,4))
sns.histplot(results_df['Residual'], kde=True)
plt.title('Residuals distribution (Actual - Predicted)')
plt.xlabel('Residual')
plt.show()


In [ ]:
# Print the coefficients and interpret them within the context of the median value prediction.
print("Intercept:", lr.intercept_)
print("\nCoefficients:")
for feature, coef in zip(X.columns, lr.coef_):
    print(f"{feature}: {coef:.2f}")

# Compute the median Sale_Price in the cleaned dataset
median_price = df_clean['Sale_Price'].median()
print(f"\nMedian SalePrice in the dataset: ${median_price:,.0f}")

# Save coefficients for interpretation
coef_liv = lr.coef_[0]
coef_gar = lr.coef_[1]

print("\nINTERPRETATION (Context of the Median Home Value):")
print(f"- Each additional square foot of **Gr_Liv_Area** increases the predicted SalePrice by "
      f"approximately **${coef_liv:.2f}**, assuming Garage_Area stays constant.")

print(f"- Each additional square foot of **Garage_Area** increases the predicted SalePrice by "
      f"approximately **${coef_gar:.2f}**, assuming Gr_Liv_Area stays constant.")

print("\nContextual meaning relative to the median home:")
print(f"- The median SalePrice is **${median_price:,.0f}**.")
print("- These coefficients tell us how much value a typical homeowner might expect to gain\n"
      "  from additional living area or garage space, compared to a median-priced house.")


# **Summarise findings**

**EDA insights:**
  **Sale_Price distribution:** Strong right-skewness with several high-value outliers. Log-transforming Sale_Price produced a more normal shape.

**Visual relationships:**
  **Gr_Liv_Area** shows a clear positive linear relationship with Sale_Price.
  **Garage_Area** also trends positively but with more scatter, indicating a weaker relationship.
  The correlations confirm that both variables are positively correlated with Sale_Price, with Gr_Liv_Area being the stronger predictor.

**Model performance:**
 Test MSE: 2,634,371,879.69
 Test RMSE: 51,326.13
 Test R²: 0.6386
 RMSE of $51,326 is large relative to the median SalePrice ($160,000), meaning the model’s absolute predictive accuracy is limited.
 R² = 0.6386 shows that the two-feature model explains about 64% of SalePrice variation, leaving substantial unexplained variance due to missing predictors

**Interpretation:**
 Gr_Liv_Area coefficient: $78.69 -> each additional sq ft increases the predicted SalePrice by about $78.69.
 Garage_Area coefficient: $141.15 -> each additional sq ft of garage increases price by about $141.15.
 Garage_area has a stronger effect per sq ft, but both predictors are meaningful.
 With only two predictors, limited R² is expected—SalePrice depends on many other influential features (quality, age, neighbourhood, total square footage, etc.).

**Recommendations:**
  Adding stronger predictors such as **Overall_Qual**, **Total_Bsmt_SF**, **Year_Built**, and neighbourhood variables.
  Considering **log-transforming Sale_Price** to reduce skewness and improve model stability.

## **OPTIONAL CHALLENGE**

In [ ]:
# Explore more potential predictors
extra_features = ['Total_Bsmt_SF', 'Year_Built', 'Full_Bath', 'Fireplaces']

# Check availability and summary statistics
df_clean[extra_features].describe()

### **Additional Feature Exploration**
I am exploring there features because they might be affecting home prices:

- **Overall_Qual** — One of the strongest predictors of SalePrice; reflects build and finish quality.
- **Total_Bsmt_SF** — More total square footage increases price; usually positively correlated.
- **Year_Built** — Newer homes tend to sell for more; may have moderate positive correlation.
- **Full_Bath** — More full bathrooms generally increase home value.
- **Fireplaces** — Often increases desirability and price, though correlation may be weaker.

In [ ]:
# Select extended set of predictors
extended_X = df_clean[['Gr_Liv_Area', 'Garage_Area', 
                       'Full_Bath', 'Total_Bsmt_SF', 'Year_Built']].copy()

y = df_clean['Sale_Price']

print("Extended feature set shape:", extended_X.shape)
extended_X.head()

In [ ]:
# Split the data using a 75:25 ratio again
X_train_ext, X_test_ext, y_train_ext, y_test_ext = train_test_split(
    extended_X, y, test_size=0.25, random_state=42)

# Fit regression model
lr_ext = LinearRegression()
lr_ext.fit(X_train_ext, y_train_ext)

# Predictions
y_pred_ext = lr_ext.predict(X_test_ext)

# Display model coefficients
print("Intercept:", lr_ext.intercept_)
print("\nCoefficients:")
pd.Series(lr_ext.coef_, index=extended_X.columns)

In [ ]:
# Compute evaluation metrics
mse_ext = mean_squared_error(y_test_ext, y_pred_ext)
rmse_ext = np.sqrt(mse_ext)
r2_ext   = r2_score(y_test_ext, y_pred_ext)

print(f"Extended Model R²: {r2_ext:.4f}")
print(f"Extended Model RMSE: {rmse_ext:,.2f}")

# Residuals
residuals_ext = y_test_ext - y_pred_ext

# Residual plot
plt.figure(figsize=(8,6))
plt.scatter(y_pred_ext, residuals_ext, alpha=0.6)
plt.axhline(0, color='red', linestyle='--', linewidth=2)
plt.xlabel("Predicted Sale_Price")
plt.ylabel("Residual (Actual - Predicted)")
plt.title("Residual Plot — Extended Model")
plt.show()

#residual distribution
plt.figure(figsize=(8,4))
sns.histplot(residuals_ext, kde=True)
plt.title("Residual Distribution — Extended Model")
plt.show()

## **Reflection: Did the model improve?**

### **Extended Model – Summary of Findings**

**Model performance:**
  **Extended Model R²:** 0.7555
  **Extended Model RMSE:** $42,219.43
  Compared to the original model (R² = 0.6386, RMSE = $51,326.13), the extended model shows **higher explanatory power** and **lower prediction error**, indicating a clear improvement.

**Interpretation of improvement:**
  Adding predictors such as **Total_Bsmt_SF**, **Year_Built**, and **Full_Bath** captures more of the true drivers of house prices.
  These features add information about overall house size, age, and functional amenities, all of which strongly influence Sale_Price.

**Residual behaviour:**
  Residuals are more tightly grouped around zero.
  There is less spread and fewer extreme errors compared to the two-variable model.
  This indicates a better-fitting model with reduced heteroscedasticity.

**Overall conclusion:**
  The extended model performs **substantially better** than the original two-predictor model.
  The reduction in RMSE by roughly **$9,100** and the increase in R² by **about 12 percentage points** show that including more relevant features significantly improves accuracy and predictive reliability.
